In [ ]:
# 📦 1. Install & Import Dependencies
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Display PyTorch and device info
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA device.")
    print("Number of GPUs:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i + 1}: {torch.cuda.get_device_name(i)}")
else:
    device = torch.device("cpu")
    print("No CUDA-compatible GPU detected. Using CPU.")

# 👇 Now you can use `device` in the rest of your notebook!


In [ ]:
# 🧹 2. Load & Normalize the Dataset

# Define a transform that converts PIL images to PyTorch tensors and scales pixel values from [0, 255] to [0.0, 1.0]
transform = transforms.ToTensor()

# Load the training portion of the MNIST dataset (60,000 images)
# - root='data': where to store or find the dataset locally
# - train=True: specifies that we want the training set
# - download=True: download the dataset if it doesn't already exist
# - transform=transform: apply the ToTensor transformation to each image
train_dataset = datasets.MNIST(root='data', train=True, download=True, transform=transform)

# Load the test portion of the MNIST dataset (10,000 images)
# - train=False: specifies that we want the test set
test_dataset = datasets.MNIST(root='data', train=False, download=True, transform=transform)

# Create a DataLoader for the training dataset
# - batch_size=64: groups the training data into batches of 64 images for more efficient training
# - shuffle=True: randomizes the order of the data each epoch to prevent overfitting and improve generalization
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Create a DataLoader for the test dataset
# - batch_size=64: evaluate the model in batches of 64
# - shuffle=False (default): ensures consistent ordering for evaluation
test_loader = DataLoader(test_dataset, batch_size=64)


In [ ]:
# 🧠 3. Define the Neural Network

# Define a simple fully connected (feedforward) neural network by subclassing nn.Module
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()  # Initialize the parent nn.Module class

        # Define the first fully connected (dense) layer
        # Takes 28x28 = 784 input features (flattened MNIST image) and outputs 128 features
        self.fc1 = nn.Linear(28 * 28, 128)

        # Define the second fully connected layer
        # Takes the 128 outputs from the previous layer and maps to 64 features
        self.fc2 = nn.Linear(128, 64)

        # Define the final fully connected layer (output layer)
        # Maps the 64 features to 10 output classes (digits 0–9)
        self.fc3 = nn.Linear(64, 10)

    # Define the forward pass — how data flows through the network
    def forward(self, x):
        # Flatten the input tensor from [batch_size, 1, 28, 28] to [batch_size, 784]
        x = x.view(-1, 28 * 28)

        # Apply ReLU activation to the output of the first layer
        x = torch.relu(self.fc1(x))

        # Apply ReLU activation to the output of the second layer
        x = torch.relu(self.fc2(x))

        # Final output layer — no activation here since loss function (CrossEntropyLoss) includes Softmax
        return self.fc3(x)

# Instantiate the model and move it to the selected device (CPU or GPU)
model = SimpleNet().to(device)


In [ ]:
# ⚙️ 4. Set Up Training Tools

# Define the loss function to use during training
# CrossEntropyLoss is suitable for multi-class classification tasks like MNIST (10 digit classes)
# It combines nn.LogSoftmax() and nn.NLLLoss() in one function, so we don't need to apply softmax manually
criterion = nn.CrossEntropyLoss()

# Set up the optimizer that updates the model's weights based on gradients
# Adam is an adaptive learning rate optimizer that generally works well for deep learning
# model.parameters() provides all the learnable weights of the model to the optimizer
optimizer = optim.Adam(model.parameters())


In [ ]:
# 🚂 5. Train the Model

# Define how many times we want the model to see the entire training dataset
epochs = 5

# Outer loop: repeat training for the specified number of epochs
for epoch in range(epochs):
    model.train()  # Set the model to training mode (enables dropout, batchnorm if used)

    running_loss = 0  # Track cumulative loss for the epoch

    # Inner loop: iterate over each batch in the training DataLoader
    for images, labels in train_loader:
        # Move both the input images and target labels to the selected device (CPU or GPU)
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()         # Reset gradients from the previous iteration
        output = model(images)        # Forward pass: compute model predictions
        loss = criterion(output, labels)  # Compute the loss between predictions and true labels
        loss.backward()              # Backward pass: compute gradients of model parameters
        optimizer.step()             # Update model parameters based on gradients

        running_loss += loss.item()  # Add this batch’s loss to the running total

    # After all batches are processed, print the loss for the current epoch
    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")


In [ ]:
# 🧪 6. Evaluate the Model

# Define a helper function to calculate classification accuracy
# - preds: the model's raw output logits
# - labels: the true target labels
# - preds.max(1): returns the index of the highest logit (predicted class) for each example
# - == labels: compares predicted vs. true labels
# - .float().mean().item(): converts match results to float, computes mean accuracy, and extracts the scalar value
def accuracy(preds, labels):
    _, pred_classes = preds.max(1)
    return (pred_classes == labels).float().mean().item()

# Switch the model to evaluation mode (disables dropout, batchnorm behavior)
model.eval()

# Variable to accumulate total accuracy over all test batches
test_acc = 0

# Turn off gradient tracking during evaluation to save memory and speed up computation
with torch.no_grad():
    # Loop over all batches in the test set
    for images, labels in test_loader:
        # Move data to the same device as the model (GPU or CPU)
        images, labels = images.to(device), labels.to(device)

        # Get model predictions (logits)
        preds = model(images)

        # Calculate and accumulate accuracy for the current batch
        test_acc += accuracy(preds, labels)

# Compute and print the average test accuracy across all batches
print(f"Test Accuracy: {test_acc / len(test_loader):.4f}")


In [ ]:
# 👁️ 7. Visualize Sample Predictions

import random  # Used to select random image indices for visualization

# Set the model to evaluation mode to ensure consistent behavior
model.eval()

# Get one full batch of test images and labels from the test DataLoader
images, labels = next(iter(test_loader))

# Move the images and labels to the appropriate device (CPU or GPU)
images, labels = images.to(device), labels.to(device)

# Turn off gradient calculation since we are only doing inference
with torch.no_grad():
    preds = model(images)              # Get model predictions (logits)
    _, pred_classes = preds.max(1)     # Extract the predicted class indices from logits

# Create a new matplotlib figure with a specific size (10 inches wide by 4 inches tall)
plt.figure(figsize=(10, 4))

# Display 6 randomly selected images from the batch
for i in range(6):
    idx = random.randint(0, len(images) - 1)  # Choose a random index from the batch

    plt.subplot(2, 3, i + 1)  # Create a subplot in a 2-row, 3-column layout
    plt.imshow(images[idx].cpu().squeeze(), cmap="gray")  # Show the image in grayscale (must move to CPU for plotting)
    
    # Add the true and predicted labels as the title
    plt.title(f"True: {labels[idx].item()} | Pred: {pred_classes[idx].item()}")
    
    # Remove axis ticks for a cleaner image
    plt.axis("off")

# Adjust spacing between subplots so they don't overlap
plt.tight_layout()

# Display the plot on screen
plt.show()
